# 02 — Keşifsel analiz, ADF ve Toda–Yamamoto

Bu defter durağanlık denetimini ve tam örneklemdeki açıklayıcı öngörü ilişkilerini gerçek veriyle hesaplar. Nedensellik tabloları yapısal neden–sonuç kanıtı değildir.

In [1]:
from pathlib import Path
import json, tomllib
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "config.toml").exists() else Path.cwd().parent
RAW, OUT = ROOT / "data/private/study-yahoo-real", ROOT / "results/research"
assert RAW.exists(), "Gerçek ham girdi data/private altında hazırlanmalıdır."
config = tomllib.loads((ROOT / "config.toml").read_text(encoding="utf-8"))
START, END, SECTORS = pd.Timestamp(config["study"]["start"]), pd.Timestamp(config["study"]["end"]), config["study"]["sectors"]
print(f"Çalışma dönemi: {START.date()} — {END.date()}")
print("Temsilciler:", ", ".join(SECTORS))

Çalışma dönemi: 2019-01-01 — 2024-12-31
Temsilciler: XBANK, XUSIN


In [2]:
from bist_risk.data import read_inputs, monthly_prices, align_macro, shock_scores
prices, macro, calendar, provenance = read_inputs(RAW, "research")
monthly = monthly_prices(prices, calendar, config["study"]["min_days"])
aligned = align_macro(macro, pd.date_range(monthly.date.min(), END, freq="ME"))
study_dates = pd.date_range(START, END, freq="ME")
quality_rows, eligible = [], []
for sector in SECTORS:
    block = monthly[(monthly.sector == sector) & monthly.date.between(START, END)].set_index("date").reindex(study_dates)
    complete = len(block) == len(study_dates) and block[["return_value", "volatility"]].notna().all().all()
    quality_rows.append({"sector": sector, "eligible": bool(complete), "months": int(block.return_value.count()), "reason": "complete" if complete else "missing daily sessions, warm-up or monthly data"})
    if complete: eligible.append(sector)
quality = pd.DataFrame(quality_rows)
assert eligible == SECTORS, quality

In [3]:
from bist_risk.modeling import supervised
frames = {s: supervised(monthly[(monthly.sector==s) & monthly.date.between(START,END)], aligned) for s in eligible}
for s, frame in frames.items(): print(s, frame.shape, "son hedef:", frame.target_date.max().date())

XBANK (72, 17) son hedef: 2025-01-31
XUSIN (72, 17) son hedef: 2025-01-31


## ADF denetimi

Düzeyler ve gerekirse farklar ADF ile incelenir. Maksimum bütünleşme derecesi, Toda–Yamamoto gecikme artırımı için kullanılır.

In [4]:
from bist_risk.econometrics import integration_order
adf_rows=[]
for sector, frame in frames.items():
    for series in ["return_value", "volatility", *aligned.columns]:
        try:
            order, records=integration_order(frame[series], config["econometrics"]["alpha"], config["econometrics"]["max_integration"])
            adf_rows.extend({"sector":sector,"series":series,"status":"ok","integration_order":order,**record} for record in records)
        except ValueError as error: adf_rows.append({"sector":sector,"series":series,"status":str(error)})
adf=pd.DataFrame(adf_rows); display(adf)

,sector,series,status,integration_order,order,statistic,p_value,lag,n
0,XBANK,return_value,ok,0,0,-8.246486,5.536301e-13,0,71
1,XBANK,volatility,ok,0,0,-5.819103,4.223884e-07,0,71
2,XBANK,brent,ok,1,0,-1.651175,4.564278e-01,0,71
3,XBANK,brent,ok,1,1,-7.182820,2.622872e-10,0,70
4,XBANK,cpi,ok,1,0,3.189967,1.000000e+00,1,70
5,XBANK,cpi,ok,1,1,-2.971617,3.763137e-02,0,70
6,XBANK,gdp_growth,ok,0,0,-3.735848,3.636259e-03,3,68
7,XBANK,industrial_production,ok,1,0,-1.921386,3.220682e-01,0,71
8,XBANK,industrial_production,ok,1,1,-9.063330,4.501332e-15,0,70
9,XBANK,policy_rate,ok,2,0,-1.662604,4.504757e-01,2,69


## Toda–Yamamoto ve çoklu test düzeltmesi

VAR düzeylerde `k + dmax` gecikmeyle kurulur. Wald kısıtları yalnızca ilk `k` gecikmeye uygulanır; BH düzeltmesi her sektör/hedef ailesi içinde yapılır.

In [5]:
from bist_risk.econometrics import family_tests
rows=[]
for sector, frame in frames.items():
    for effect in ["return_value","volatility"]:
        tests=family_tests(frame, list(aligned.columns), effect, config["econometrics"])
        tests["sector"], tests["scope"] = sector, "full_sample_descriptive_not_feature_selection"
        rows.append(tests)
causality=pd.concat(rows, ignore_index=True); display(causality)

,cause,effect,k,dmax,n,statistic,p_value,whiteness_p,residual_ok,status,q_value,sector,scope
0,brent,return_value,1,1,70,0.011052,0.916275,0.528631,True,ok,0.916275,XBANK,full_sample_descriptive_not_feature_selection
1,cpi,return_value,2,1,69,3.256221,0.196300,0.788554,True,ok,0.303061,XBANK,full_sample_descriptive_not_feature_selection
2,gdp_growth,return_value,1,0,71,2.469278,0.116091,0.000445,False,ok,0.303061,XBANK,full_sample_descriptive_not_feature_selection
3,industrial_production,return_value,1,1,70,0.471265,0.492406,0.202519,True,ok,0.590887,XBANK,full_sample_descriptive_not_feature_selection
4,policy_rate,return_value,3,2,67,4.617565,0.202040,0.705977,True,ok,0.303061,XBANK,full_sample_descriptive_not_feature_selection
5,usdtry,return_value,1,1,70,4.431095,0.035290,0.268689,True,ok,0.211739,XBANK,full_sample_descriptive_not_feature_selection
6,brent,volatility,1,1,70,2.109321,0.146404,0.194817,True,ok,0.836326,XBANK,full_sample_descriptive_not_feature_selection
7,cpi,volatility,2,1,69,0.480329,0.786498,0.426236,True,ok,0.943798,XBANK,full_sample_descriptive_not_feature_selection
8,gdp_growth,volatility,1,0,71,0.285801,0.592924,0.000079,False,ok,0.889386,XBANK,full_sample_descriptive_not_feature_selection
9,industrial_production,volatility,1,1,70,0.003625,0.951988,0.231653,True,ok,0.951988,XBANK,full_sample_descriptive_not_feature_selection


In [6]:
ok=causality[causality.status.eq("ok")]
fig=px.bar(ok,x="cause",y="q_value",color="sector",facet_row="effect",barmode="group",title="Toda–Yamamoto: BH düzeltilmiş q değerleri")
fig.add_hline(y=config["econometrics"]["alpha"],line_dash="dash",line_color="red"); fig.show()

In [7]:
from bist_risk.artifacts import write_tables
write_tables(OUT,{"adf":adf,"causality":causality}); print("02 ekonometrik tabloları kaydedildi.")

02 ekonometrik tabloları kaydedildi.
